## Лабораторная работа № 1 
## Выполнение разведочного анализа больших данных с использованием фреймворка Apache Spark

### Часть 1

В данной части работы рассмотрены:
* загрузка данных из HDFS;
* базовые преобразования данных;
* загрузка преобразованных данных в таблицу `Apache Airflow`.

Подключим необходимые библиотеки.

In [1]:
import os
from pyspark.sql import SparkSession, DataFrame
from pyspark import SparkConf
from pyspark.sql.functions import (
    regexp_replace,
    regexp_extract_all,
    regexp_extract,
    col,
    lit,
    when,
    to_date,
    from_unixtime,
    split,
    trim,
    year
)

Сформируем объект конфигурации для `Apache Spark`, указав необходимые параметры.

In [2]:
def create_spark_configuration() -> SparkConf:
    """
    Создает и конфигурирует экземпляр SparkConf для приложения Spark.

    Returns:
        SparkConf: Настроенный экземпляр SparkConf.
    """
    # Получаем имя пользователя
    # Получаем имя пользователя
    user_name = os.getenv("USER")
    
    conf = SparkConf()
    conf.setAppName("Lab_1_DC")
    conf.setMaster("local[*]")
    # conf.set("spark.submit.deployMode", "client")
    conf.set("spark.executor.memory", "12g")
    conf.set("spark.executor.cores", "6")
    conf.set("spark.executor.instances", "1")
    conf.set("spark.driver.memory", "4g")
    conf.set("spark.driver.cores", "1")
    conf.set("spark.sql.catalog.spark_catalog.type", "hadoop")
    conf.set("spark.sql.catalog.spark_catalog.warehouse", f"hdfs:///b")
    conf.set("spark.sql.catalog.spark_catalog.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")

    return conf

Создаём сам объект конфигурации.

In [3]:
conf = create_spark_configuration()

Создаём и выводим на экран сессию `Apache Spark`. В процессе создания сессии происходит подключение к кластеру `Apache Hadoop`, что может занять некоторое время.

In [4]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark

Для исследования будем использовать датасет `"Big Sales Data"`, расположенный на платформе `Kaggle` по адресу https://www.kaggle.com/datasets/pigment/big-sales-data/data.

Указываем путь.

In [5]:
path1 = 'hdfs://namenode:9000/b/Books_rating.csv'
path2 = 'hdfs://namenode:9000/b/books_data.csv'

Заполняем датафрейм данными из файла.

In [6]:
df1 = (spark.read.format("csv")
      .option("header", "true")
      .load(path1)
)

df2 = (spark.read.format("csv")
      .option("header", "true")
      .load(path2)
)

df = df1.join(df2, "Title", "inner")

Выводим фрагмент датафрейма на экран.

In [7]:
df.limit(20).toPandas().style\
    .set_properties(**{'text-align': 'left', 'max-width': '0', 'white-space': 'nowrap', 'overflow': 'hidden', 'text-overflow': 'ellipsis'})\
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])\
    .format(precision=2)

,Title,Id,Price,User_id,profileName,review/helpfulness,review/score,review/time,review/summary,review/text,description,authors,image,previewLink,publisher,publishedDate,infoLink,categories,ratingsCount
0,"""""""Always ready!"""": The story of the United States Coast guard",B0007H2HAM,""",,A3AUL23GMCOP2A,Andrew S. Rogers,1/1,2.0,1156032000,""""""Sound bodies",stout hearts,"and alert minds""""""","""It's hard to believe that any history of the U.S. Coast Guard published in 1944 would have much to recommend it today. Too much of significance has happened in the years since. If """"Always Ready!"""" does have anything in its favor",it's the narrative ability of its author,"Kensil Bell.By the time """"Always Ready!"""" was published",Bell had already written a couple of novels about the Coast Guard. I was impressed not long ago with the second of these (Coast Guard Cadets,1941) which was not only an Academy novel -- a particular interest of mine -- but one where the Academy setting was actually relevant to the story,""",,['Kensil Bell'],http://books.google.com/books/content?id=ElUEAAAAMAAJ&printsec=frontcover&img=1&zoom=1&source=gbs_api,""http://books.google.com/books?id=ElUEAAAAMAAJ&q=%22Always+ready!%22:+The+story+of+the+United+States+Coast+guard",&dq=%22Always+ready!%22:+The+story+of+the+United+States+Coast+guard,"&hl=&cd=1&source=gbs_api""",None,1943,"http://books.google.com/books?id=ElUEAAAAMAAJ&dq=%22Always+ready!%22:+The+story+of+the+United+States+Coast+guard,&hl=&source=gbs_api",None,None,None
1,"""""""Carefree"""" (R.K.O.Classic Screenplays)""",0804467900,None,A15LK8DSFQZZ52,"""Patricia R. Andersen """"redheaded booklover""""""",0/0,5.0,1045526400,"Fred and Ginger, I remember you","This book is the screenplay for Ginger Rogers and Fred Astaire's final Irving Berlin film musical. The premise is that Ginger Rogers is engaged to Ralph Bellamy but she keeps postponing the marriage. Mr Bellamy sends her to his good friend, Fred Astaire, who happens to be a psychiatrist. True to screwball comedies of the time, M's Rogers falls madly in love with Mr Astaire. Mr Astaire plants a post hypnotic suggestion so M's Rogers will hate him and lover her fiance, Mr Bellamy. But, of course, Mr Astaire has fallen for M's Rogers and wants to reverse the suggestion.Besides the complete screenplay, there are also many black and white photographs of the film. This is quite a good book for any Fred Astaire or Ginger Rogers fans and I recommend it highly.",None,"['Allan Scott', 'Ernest Pagano']",None,http://books.google.com/books?id=GtHkuQEACAAJ&dq=%22Carefree%22+(R.K.O.Classic+Screenplays)&hl=&cd=1&source=gbs_api,None,1985,http://books.google.com/books?id=GtHkuQEACAAJ&dq=%22Carefree%22+(R.K.O.Classic+Screenplays)&hl=&source=gbs_api,None,None
2,"""""""Catch 'em alive Jack"""";: The life and adventures of an American pioneer",B00085T7O2,""",,ASEH0CVYVGZCZ,""dagmara """"dagmara""""""",0/1,5.0,1341705600,Fascinating life and historical facts,"There are many ideas what life was like in the Old West, but Jack Abernathy's biography is more exciting than anyone can imagine. Rather than just talk about himself, there are great vignettes of life at that time: riding on a cattle drive, wildcatting, and hunting wolves by hand.To complete this story, buy Bud and Me, the exciting true story of his two sons, Louis and Bud Abernathy, and their ride from Oklahoma to New Mexico ALONE as well as from Oklahoma to New York City. ALONE.The combined books provide a very complete picture of life at the turn of the century, when the Old West was fading and technology was creeping in.",None,None,""",""Best known for catching wolves alive with his bare hands",John R. Abernathy (1876?1941) was born to Scottish ancestors in Texas. Raised in the burgeoning railroad town of Sweetwater,Abernathy considered himself a true son of the Wild West. In his amazing life he worked as a U.S. marshal,sheriff,Secret Service agent,and wildcat oil driller. But it was the accidental discovery 

Очевидно, что в целях сохранения ясности изложения и сокращения расчетного времени имеет смысл рассматривать не все столбцы датасета. Оставим следующие колонки, удалив остальные:

### Анализ отзывов и рейтингов
| Название столбца | Расшифровка |
| -------- | ------ |
| id |	Уникальный идентификатор книги (ISBN/ASIN)
| title	| Название книги
| review_score	| Оценка книги пользователем (1.0-5.0)
| review_text	| Текст отзыва
| review_summary	| Заголовок отзыва
| review_helpfulness	| Полезность отзыва (в формате "X/Y")
| review_time	| Временная метка отзыва

### Анализ книг и метаданных
| Название столбца | Расшифровка |
| -------- | ------ |
| authors	| Авторы книги
| publisher	| Издательство
| published_date	| Дата публикации книги
| categories	| Категории/жанры книги

In [8]:
df = df.select(
    "Id", "Title", "authors", "publisher", "publishedDate", "categories",
    "review/score", "review/text", "review/summary", "review/helpfulness",
    "review/time", "User_id", "profileName"
)

In [9]:
df.limit(20).toPandas().style\
    .set_properties(**{'text-align': 'left', 'max-width': '0', 'white-space': 'nowrap', 'overflow': 'hidden', 'text-overflow': 'ellipsis'})\
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])\
    .format(precision=2)

,Id,Title,authors,publisher,publishedDate,categories,review/score,review/text,review/summary,review/helpfulness,review/time,User_id,profileName
0,B0007H2HAM,"""""""Always ready!"""": The story of the United States Coast guard",&dq=%22Always+ready!%22:+The+story+of+the+United+States+Coast+guard,1943,"http://books.google.com/books?id=ElUEAAAAMAAJ&dq=%22Always+ready!%22:+The+story+of+the+United+States+Coast+guard,&hl=&source=gbs_api",None,it's the narrative ability of its author,1941) which was not only an Academy novel -- a particular interest of mine -- but one where the Academy setting was actually relevant to the story,Bell had already written a couple of novels about the Coast Guard. I was impressed not long ago with the second of these (Coast Guard Cadets,"""It's hard to believe that any history of the U.S. Coast Guard published in 1944 would have much to recommend it today. Too much of significance has happened in the years since. If """"Always Ready!"""" does have anything in its favor","Kensil Bell.By the time """"Always Ready!"""" was published",stout hearts,"and alert minds"""""""
1,0804467900,"""""""Carefree"""" (R.K.O.Classic Screenplays)""","['Allan Scott', 'Ernest Pagano']",None,1985,None,5.0,"This book is the screenplay for Ginger Rogers and Fred Astaire's final Irving Berlin film musical. The premise is that Ginger Rogers is engaged to Ralph Bellamy but she keeps postponing the marriage. Mr Bellamy sends her to his good friend, Fred Astaire, who happens to be a psychiatrist. True to screwball comedies of the time, M's Rogers falls madly in love with Mr Astaire. Mr Astaire plants a post hypnotic suggestion so M's Rogers will hate him and lover her fiance, Mr Bellamy. But, of course, Mr Astaire has fallen for M's Rogers and wants to reverse the suggestion.Besides the complete screenplay, there are also many black and white photographs of the film. This is quite a good book for any Fred Astaire or Ginger Rogers fans and I recommend it highly.","Fred and Ginger, I remember you",0/0,1045526400,A15LK8DSFQZZ52,"""Patricia R. Andersen """"redheaded booklover"""""""
2,B00085T7O2,"""""""Catch 'em alive Jack"""";: The life and adventures of an American pioneer",John R. Abernathy (1876?1941) was born to Scottish ancestors in Texas. Raised in the burgeoning railroad town of Sweetwater,Secret Service agent,and wildcat oil driller. But it was the accidental discovery of a bold means of catching wolves alive that made Abernathy famous and drew the attention of President Theodore Roosevelt. By forcing his hand deep enough into a wolf's mouth,"a service for which he was paid fifty dollars by eager ranchers. ø This Bison Books edition brings Abernathy's vivid account of his life into print for the first time since its original publication in 1936.""",Fascinating life and historical facts,None,None,1341705600,"There are many ideas what life was like in the Old West, but Jack Abernathy's biography is more exciting than anyone can imagine. Rather than just talk about himself, there are great vignettes of life at that time: riding on a cattle drive, wildcatting, and hunting wolves by hand.To complete this story, buy Bud and Me, the exciting true story of his two sons, Louis and Bud Abernathy, and their ride from Oklahoma to New Mexico ALONE as well as from Oklahoma to New York City. ALONE.The combined books provide a very complete picture of life at the turn of the century, when the Old West was fading and technology was creeping in.",0/1,5.0
3,B00085T7O2,"""""""Catch 'em alive Jack"""";: The life and adventures of an American pioneer",John R. Abernathy (1876?1941) was born to Scottish ancestors in Texas. Raised in the burgeoning railroad town of Sweetwater,Secret Service agent,and wildcat oil driller. But it was the accidental discovery of a bold means of catching wolves alive that made Abernathy famous and drew the attention of President Theodore Roosevelt. By forcing his hand deep enough into a wolf's mouth,"a service for which he w

Выведем на экран метаданные датасета.

In [10]:
df.printSchema()

root
 |-- Id: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- publishedDate: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- review/score: string (nullable = true)
 |-- review/text: string (nullable = true)
 |-- review/summary: string (nullable = true)
 |-- review/helpfulness: string (nullable = true)
 |-- review/time: string (nullable = true)
 |-- User_id: string (nullable = true)
 |-- profileName: string (nullable = true)



Видно, что все столбцы датасета содержат строковый тип данных, что не соответствует ожиданиям. Выполним преобразования типов данных некоторых столбцов.

In [11]:
def transform_dataframe(data: DataFrame) -> DataFrame:
    """
    Преобразует столбцы DataFrame в указанные типы данных и
    выполняет необходимые преобразования.

    Args:
        data (DataFrame): Исходный DataFrame.

    Returns:
        DataFrame: Преобразованный DataFrame.
    """
    data = data \
        .withColumnRenamed("Id", "id") \
        .withColumnRenamed("Title", "title") \
        .withColumnRenamed("review/time", "review_time") \
        .withColumnRenamed("review/helpfulness", "review_helpfulness") \
        .withColumnRenamed("review/summary", "review_summary") \
        .withColumnRenamed("review/text", "review_text") \
        .withColumnRenamed("review/score", "review_score") \
        .withColumnRenamed("publishedDate", "published_date") \
        .withColumnRenamed("User_id", "user_id") \
        .withColumnRenamed("profileName", "profile_name")
    
    # Преобразуем столбцы в соответствующие типы данных
    data = data \
        .withColumn("id", col("id").cast("string")) \
        .withColumn("review_score", col("review_score").cast("double")) \
        .withColumn("review_time", col("review_time").cast("integer")) \
        .withColumn("user_id", col("user_id").cast("string"))
    
    # Извлекаем числовую часть из review_helpfulness
    data = data.withColumn(
        "helpfulness_numerator", 
        regexp_extract(col("review_helpfulness"), r"(\d+)/\d+", 1).cast("integer")
    ).withColumn(
        "helpfulness_denominator", 
        regexp_extract(col("review_helpfulness"), r"\d+/(\d+)", 1).cast("integer")
    ).withColumn(
        "helpfulness_ratio", 
        when(col("helpfulness_denominator") > 0, 
             col("helpfulness_numerator") / col("helpfulness_denominator"))
        .otherwise(0.0)
    )
    
    # Преобразуем publishedDate в дату (обрабатываем разные форматы)
    data = data.withColumn(
    "published_date_clean",
    when(col("published_date").rlike(r"^\\d{4}$"),  # только год
         to_date(lit("01-01-") + col("published_date"), "dd-MM-yyyy"))
    .when(col("published_date").rlike(r"^\\d{4}-\\d{2}-\\d{2}$"),  # YYYY-MM-DD
         to_date(col("published_date"), "yyyy-MM-dd"))
    .otherwise(None)
)

    # Преобразуем authors и categories в массивы (если они в строковом формате списка)
    data = data.withColumn(
        "authors_array",
        when(col("authors").rlike(r"^\[.*\]$"),
             regexp_extract_all(col("authors"), lit(r"'([^']*)'"), 1))
        .otherwise(split(col("authors"), ","))
    )
    
    data = data.withColumn(
        "categories_array",
        when(col("categories").rlike(r"^\[.*\]$"),
             regexp_extract_all(col("categories"), lit(r"'([^']*)'"), 1))
        .otherwise(split(col("categories"), ","))
    )
    
    # Очистка текстовых полей от лишних пробелов и специальных символов
    text_columns = ["title", "publisher", "review_summary", "review_text", "profile_name"]
    
    for col_name in text_columns:
        data = data.withColumn(
            col_name,
            when(col(col_name).isNotNull(), 
                 trim(regexp_replace(col(col_name), r"\s+", " ")))
            .otherwise(col(col_name))
        )
    
    # Создаем год публикации для анализа
    data = data.withColumn(
        "published_year",
        year(col("published_date_clean"))
    )
    
    return data


In [12]:
df = transform_dataframe(df)

Очистка

In [13]:
cleaned_df = df.fillna({
    "authors": "Unknown",
    "publisher": "Unknown", 
    "categories": "Unknown",
    "review_text": "",
    "review_summary": "",
    "profile_name": "Anonymous"
}).fillna(0)  # для числовых колон

df = cleaned_df

In [14]:
df.limit(20).toPandas().style\
    .set_properties(**{'text-align': 'left', 'max-width': '0', 'white-space': 'nowrap', 'overflow': 'hidden', 'text-overflow': 'ellipsis'})\
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])\
    .format(precision=2)

,id,title,authors,publisher,published_date,categories,review_score,review_text,review_summary,review_helpfulness,review_time,user_id,profile_name,helpfulness_numerator,helpfulness_denominator,helpfulness_ratio,published_date_clean,authors_array,categories_array,published_year
0,B0007H2HAM,"""""""Always ready!"""": The story of the United States Coast guard",&dq=%22Always+ready!%22:+The+story+of+the+United+States+Coast+guard,1943,"http://books.google.com/books?id=ElUEAAAAMAAJ&dq=%22Always+ready!%22:+The+story+of+the+United+States+Coast+guard,&hl=&source=gbs_api",Unknown,0.00,1941) which was not only an Academy novel -- a particular interest of mine -- but one where the Academy setting was actually relevant to the story,Bell had already written a couple of novels about the Coast Guard. I was impressed not long ago with the second of these (Coast Guard Cadets,"""It's hard to believe that any history of the U.S. Coast Guard published in 1944 would have much to recommend it today. Too much of significance has happened in the years since. If """"Always Ready!"""" does have anything in its favor",0,stout hearts,"and alert minds""""""",0,0,0.00,None,['&dq=%22Always+ready!%22:+The+story+of+the+United+States+Coast+guard'],None,0
1,0804467900,"""""""Carefree"""" (R.K.O.Classic Screenplays)""","['Allan Scott', 'Ernest Pagano']",Unknown,1985,Unknown,5.00,"This book is the screenplay for Ginger Rogers and Fred Astaire's final Irving Berlin film musical. The premise is that Ginger Rogers is engaged to Ralph Bellamy but she keeps postponing the marriage. Mr Bellamy sends her to his good friend, Fred Astaire, who happens to be a psychiatrist. True to screwball comedies of the time, M's Rogers falls madly in love with Mr Astaire. Mr Astaire plants a post hypnotic suggestion so M's Rogers will hate him and lover her fiance, Mr Bellamy. But, of course, Mr Astaire has fallen for M's Rogers and wants to reverse the suggestion.Besides the complete screenplay, there are also many black and white photographs of the film. This is quite a good book for any Fred Astaire or Ginger Rogers fans and I recommend it highly.","Fred and Ginger, I remember you",0/0,1045526400,A15LK8DSFQZZ52,"""Patricia R. Andersen """"redheaded booklover""""""",0,0,0.00,None,"['Allan Scott', 'Ernest Pagano']",None,0
2,B00085T7O2,"""""""Catch 'em alive Jack"""";: The life and adventures of an American pioneer",John R. Abernathy (1876?1941) was born to Scottish ancestors in Texas. Raised in the burgeoning railroad town of Sweetwater,Secret Service agent,and wildcat oil driller. But it was the accidental discovery of a bold means of catching wolves alive that made Abernathy famous and drew the attention of President Theodore Roosevelt. By forcing his hand deep enough into a wolf's mouth,"a service for which he was paid fifty dollars by eager ranchers. ø This Bison Books edition brings Abernathy's vivid account of his life into print for the first time since its original publication in 1936.""",0.00,,,1341705600,0,0/1,5.0,0,0,0.00,None,[' John R. Abernathy (1876?1941) was born to Scottish ancestors in Texas. Raised in the burgeoning railroad town of Sweetwater'],"[' a service for which he was paid fifty dollars by eager ranchers. ø This Bison Books edition brings Abernathy\'s vivid account of his life into print for the first time since its original publication in 1936.""']",0
3,B00085T7O2,"""""""Catch 'em alive Jack"""";: The life and adventures of an American pioneer",John R. Abernathy (1876?1941) was born to Scottish ancestors in Texas. Raised in the burgeoning railroad town of Sweetwater,Secret Service agent,and wildcat oil driller. But it was the accidental discovery of a bold means of catching wolves alive that made Abernathy famous and drew the attention of President Theodore Roosevelt. By forcing his hand deep enough into a wolf's mouth,"a service for which he was paid fifty dollars by eager ranchers. ø This Bison Books edition brings Abernathy's vivid account of his life int

In [15]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- authors: string (nullable = false)
 |-- publisher: string (nullable = false)
 |-- published_date: string (nullable = true)
 |-- categories: string (nullable = false)
 |-- review_score: double (nullable = false)
 |-- review_text: string (nullable = false)
 |-- review_summary: string (nullable = false)
 |-- review_helpfulness: string (nullable = true)
 |-- review_time: integer (nullable = true)
 |-- user_id: string (nullable = true)
 |-- profile_name: string (nullable = false)
 |-- helpfulness_numerator: integer (nullable = true)
 |-- helpfulness_denominator: integer (nullable = true)
 |-- helpfulness_ratio: double (nullable = false)
 |-- published_date_clean: date (nullable = true)
 |-- authors_array: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- categories_array: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- published_year: integer (nullable = t

# Сохранение DataFrame в формате Parquet

### Запись данных из Parquet

In [16]:
database_name = "vasilev_database"
table_name = "books_table"

output_path = f"hdfs://namenode:9000/b/{database_name}/{table_name}"

print(f"Сохраняем данные в: {output_path}")

df.write.mode("overwrite").option("compression", "snappy").parquet(output_path)

print("Данные успешно сохранены в Parquet формате в HDFS")

Сохраняем данные в: hdfs://namenode:9000/b/vasilev_database/books_table
Данные успешно сохранены в Parquet формате в HDFS


### Чтение данных из Parquet

In [17]:
df_parquet = spark.read.parquet(output_path)

print("Данные успешно загружены из Parquet")
print(f"Количество строк: {df_parquet.count()}")
print(f"Количество колонок: {len(df_parquet.columns)}")

# Показать схему данных
print("Схема данных:")
df_parquet.printSchema()

# Показать первые несколько строк
print("Первые 10 строк:")

df_parquet.limit(10).toPandas().style\
    .set_properties(**{'text-align': 'left', 'max-width': '0', 'white-space': 'nowrap', 'overflow': 'hidden', 'text-overflow': 'ellipsis'})\
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])\
    .format(precision=2)

Данные успешно загружены из Parquet
Количество строк: 2999829
Количество колонок: 20
Схема данных:
root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- published_date: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- review_score: double (nullable = true)
 |-- review_text: string (nullable = true)
 |-- review_summary: string (nullable = true)
 |-- review_helpfulness: string (nullable = true)
 |-- review_time: integer (nullable = true)
 |-- user_id: string (nullable = true)
 |-- profile_name: string (nullable = true)
 |-- helpfulness_numerator: integer (nullable = true)
 |-- helpfulness_denominator: integer (nullable = true)
 |-- helpfulness_ratio: double (nullable = true)
 |-- published_date_clean: date (nullable = true)
 |-- authors_array: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- categories_array: array (nullable = true)

,id,title,authors,publisher,published_date,categories,review_score,review_text,review_summary,review_helpfulness,review_time,user_id,profile_name,helpfulness_numerator,helpfulness_denominator,helpfulness_ratio,published_date_clean,authors_array,categories_array,published_year
0,1891882007,"""""""Breaking Traditions"""" Cathedral Windows Quick Method Quilt""","['Susan Tedrow Fisher', 'Kimberly Nappier']",Unknown,1999,['Patchwork'],4.00,"This is an interesting twist on a time-honored quilt pattern. Although I have always liked the cathedral windows pattern, I have always resisted making one due to the tremendous amount of hand sewing. This book allows you to create a beautiful quilt using machine techniques. The instructions are brief, to-the-point and clear. The illustrations appear to be hand-drawn, but they serve to explain the process and do not detract from the text. The net result is a beautiful old-style quilt streamlined for those of us who do not have the time to create it all by hand. My next full-sized project will be a cathedral window quilt.",Breaking traditions cathedral windows quick method quilt,59/61,960768000,A124LUSIP65B1U,School Librarian,59,61,0.97,None,"['Susan Tedrow Fisher', 'Kimberly Nappier']",['Patchwork'],0
1,1891882007,"""""""Breaking Traditions"""" Cathedral Windows Quick Method Quilt""","['Susan Tedrow Fisher', 'Kimberly Nappier']",Unknown,1999,['Patchwork'],3.00,"""This is not a book. It is 12 pages of instructions with hand drawn illustrations of the block units. If all you want are instructions on """"How to"""" make a traditional Cathedral Window block",Quick & Easy Instruction Sheet,31/32,1039564800,A3Q54OUHBC872L,"""G. W. Coleman """"IT Specialist""""""",31,32,0.97,None,"['Susan Tedrow Fisher', 'Kimberly Nappier']",['Patchwork'],0
2,1891882007,"""""""Breaking Traditions"""" Cathedral Windows Quick Method Quilt""","['Susan Tedrow Fisher', 'Kimberly Nappier']",Unknown,1999,['Patchwork'],1.00,"The price of this book is about 8 times what it is worth. First of all, it is not a ""paperback""; it is a ""pamphlet"". There is no firm cover or back. I feel taken advantage of when I order a book expecting a book and get a pamphlet. Some pages held together and called a ""paperback""!!!!!!!!!!!!!!!!!!!!",Got to be kidding,18/20,1083974400,A3AY8G2GT95RHL,Jacqueline Rund,18,20,0.90,None,"['Susan Tedrow Fisher', 'Kimberly Nappier']",['Patchwork'],0
3,1891882007,"""""""Breaking Traditions"""" Cathedral Windows Quick Method Quilt""","['Susan Tedrow Fisher', 'Kimberly Nappier']",Unknown,1999,['Patchwork'],5.00,"""I loved this """"book."""" The directions are excellent. The drawings are precise and provide the perfect compliment to the written directions. If you've ever wanted to make a cathedral window quilt",Horse Rider,1/1,1222041600,A1DV53Z7XEJEY7,"""Horse Rider """"ET""""""",1,1,1.00,None,"['Susan Tedrow Fisher', 'Kimberly Nappier']",['Patchwork'],0
4,B000PH26B4,"""""""Dear America"""" Four Volume Boxed Set (Dear America)""",Dear America: Standing in the Light,['Scholastic Books'],None,Unknown,5.00,my daughter will be so surprised on christmas morning and im so glad i found what i was looking for for such a good price. I recieved it and its perfect!,happy,0/0,1354924800,A2D99F1RC3ZCVI,cindy bancroft,0,0,0.00,None,[' Dear America: Standing in the Light'],None,0
5,0898702755,"""""""Forget Not Love"""": The Passion of Maximilian Kolbe""",['André Frossard'],Ignatius Press,1991,['Religion'],5.00,"""Forget Not Love"" is the story of St. Maximilian Kolbe, the Polish Franciscan who offered to die in the place of a married man at Auschwitz, yet this book is about so much more. It creates a portrait of Kolbe as a real human being, it tells of one man's zeal for his faith and country, and it is about what love really is. This book is one of the best I have read on St. Maximilian. Frossard's beautiful writing is an added plus. I HIGHLY reccomend this book to all!","""Forget Not Love"" is Special",43/44,932342400,None

После успешной записи таблицы останавливаем сессию `Apache Spark`.

In [18]:
spark.stop()